## Graph construction

- Construct NN-graphs w/ $G(V,E)$ with nodes $v_i \in V$ as DAPI coordinates & edges $(e_i, e_j) \in E$ as affinity btw adjacent nodes

- Graph comparisons btw L-slices w/ H&E channels, CyIF channels & joint channels

In [ ]:
import os
import gc
import sys
import pickle

import numpy as np
import cupy as cp
import pandas as pd

import networkx as nx
import tifffile

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

import torch
torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from cellpose import utils as cp_utils
from cellpose import plot as cp_plot
# from cellpose.models import CellposeModel

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
import IO, utils, vgae

### Segmentation & Graph Construction

#### CyIF channels

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CellposeModel(
    gpu=device,
    model_type='nuclei',
    # pretrained_model=pretrained_model.value
)

In [ ]:
#  Load dataset
data_path = '../data/cycif/zonation_hires/'
filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']
dapis = [tifffile.imread(os.path.join(data_path, f))[0] for f in filenames]
gc.collect()


In [ ]:
out_path = '../data/cycif/segment/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

for f, img in zip(filenames, dapis):
    print('Segmentation on image {}...'.format(f))
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min())

    mask = model.eval(
        img,
        diameter=8,
        tile=True,
        channels=[0, 0],
        flow_threshold=1,
        min_size=50
    )[0]
    torch.cuda.empty_cache()

    tifffile.imwrite(os.path.join(out_path, f.split('.')[0] + '_mask.tif'), mask)
    

Subsampling nodes & construct networks:

In [ ]:
img_path = '../data/cycif/zonation_hires/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']
dapis = [tifffile.imread(os.path.join(img_path, f))[0]
        for f in filenames]
masks = [tifffile.imread(os.path.join(mask_path, f.split('.')[0]+'_mask.tif'))
         for f in filenames]


In [ ]:
def get_centroids(mask):
    mempool = cp.get_default_memory_pool()

    mask_cp = cp.array(mask)
    coords = []
    labels = cp.unique(mask_cp)[1:]
    for label in labels:
        pts_y, pts_x = cp.nonzero(mask_cp == label)
        cy = cp.round(pts_y.mean()).astype(cp.uint32)
        cx = cp.round(pts_x.mean()).astype(cp.uint32)
        coords.append([int(cy.get()), int(cx.get())])
        mempool.free_all_blocks()    

    return np.array(coords)


def get_coords_from_graph(G, node_list=None):
    try:
        nodes = G.nodes if node_list is None else node_list
        return np.array([
            [G.nodes[n]['pos'][1], G.nodes[n]['pos'][0]]
             for n in nodes
        ])
    except KeyError as e:
        print(e, 'Please assign `pos` to graph nodes first')


def construct_graph(coords, k=5, r=30):
    G = nx.Graph()
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(coords)
    distances, nn_indices = nbrs.kneighbors(coords)
    distances, nn_indices = distances[:, 1:], nn_indices[:, 1:]  # Skip self

    # Construct Graph w/ all nodes
    for i, (y, x) in enumerate(coords):
        G.add_node(i, pos=(x, y))  # XY-based coords

    # Add edges within `r` resolution
    for i in range(len(coords)):
        for j, distance in zip(nn_indices[i], distances[i]):
            if i != j and (r == np.inf or distance <= r):
                G.add_edge(i, j, weight=1/distance)

    return G


def sample_nodes(G, k=5, r=np.inf, 
                 sample_ratio=0.5, res=0.5):
    partition = nx.community.louvain_communities(G, resolution=res)
    sampled_nodes = []
    for nodes in partition:
        sample_size = np.ceil(len(nodes)*sample_ratio).astype(np.uint8)
        sampled_nodes.extend(np.random.choice(list(nodes), size=sample_size, replace=False))
    coords = get_coords_from_graph(G, sampled_nodes)
    G_new = construct_graph(coords, k=k, r=r)
    return G_new


In [ ]:
# Save simplified "meta-graphs" per L-slice
out_path = '../data/cycif/graph_new/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

prop_nodes_to_keep = 0.1

for f, mask in zip(filenames, masks):
    print('Computing graph for {}...'.format(f))
    coords = get_centroids(mask)
    G_raw = construct_graph(coords, k=5, r=30)
    G = sample_nodes(G_raw, sample_ratio=0.1, k=3)
    pickle.dump(G, open(os.path.join(out_path, f.split('.')[0] + '.pkl'), 'wb'))

    del G_raw, G
    gc.collect()

In [ ]:
Gs = []
for f, dapi in zip(filenames, dapis):
    G =  pickle.load(open(os.path.join(out_path, f.split('.')[0] + '.pkl'), 'rb'))
    Gs.append(G)
    del G, dapi

In [ ]:
L = len(Gs)
n_nodes = [len(G) for G in Gs]

fig, ax = plt.subplots(figsize=(8, 2))
ax.plot(np.arange(L), n_nodes, '.-')
ax.set_xlabel('# section')
ax.set_ylabel(r"$\vert V \vert$")

plt.show()

del L

In [ ]:
def disp_network(img, G, figsize=None,
                 node_size=5, edge_width=1,
                 title=None):
    pos = {n: G.nodes[n]['pos'] for n in G.nodes}
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap='magma', alpha=0.5)
    ax.axis('off')
    nx.draw_networkx_nodes(G, pos, node_color='yellow', node_size=node_size, ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='lime', width=edge_width, ax=ax)
    plt.tight_layout()
    plt.title(title, fontsize=fontsize)

def generate_random_colors(n):
    random_colors = []
    for _ in range(n):
        # Generate random RGB values
        r = np.random.randint(0, 255)
        g = np.random.randint(0, 255)
        b = np.random.randint(0, 255)
        # Convert RGB values to hexadecimal color code
        color_code = "#{:02x}{:02x}{:02x}".format(r, g, b)
        random_colors.append(color_code)
    return random_colors


def disp_graph_overlaps(Gs, labels, figsize,
                        node_size=5, edge_width=1,
                        title=None):
    colors = generate_random_colors(len(Gs))
    fig, ax = plt.subplots(figsize=figsize)
    for c, lbl, G in zip(colors, labels, Gs):
        pos = nx.get_node_attributes(G, 'pos')
        nx.draw_networkx_nodes(G, pos, node_color=c, node_size=node_size, label=lbl, ax=ax)
        nx.draw_networkx_edges(G, pos, edge_color=c, width=edge_width, ax=ax)

    plt.legend()
    plt.tight_layout()
    plt.title(title, fontsize=20)
    plt.show() 
    


In [ ]:
disp_graph_overlaps(Gs[3:6], labels=filenames[3:6], figsize=(8, 8), 
                    node_size=0.5, edge_width=0.1)

Feature matrix extraction

In [ ]:
cyif_chans = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68'
]

In [ ]:
img_path = '../data/cycif/zonation_hires/'
graph_path = '../data/cycif/graph/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']

# Load images
imgs_rgb = [tifffile.imread(os.path.join(img_path, f))
            for f in filenames]

# Load graphs
Gs = [pickle.load(open(os.path.join(graph_path, f), 'rb'))
      for f in sorted(os.listdir(graph_path))
      if f[-3:] == 'pkl']

# Retrieve graph coords
coords_list = [get_coords_from_graph(G) for G in Gs]

In [ ]:
# Convert channel expressions to [0, 1]
# Covert empty channels to 0s
imgs = []
for img in imgs_rgb:
    img_adj = img.copy().astype(np.float32)
    for i, chan in enumerate(img):
        if chan.max() <= 1:
            img_adj[i] = 0
        else:
            img_adj[i] = (img_adj[i] - img_adj[i].min()) / (img_adj[i].max() - img_adj[i].min())
    imgs.append(img_adj)
del imgs_rgb

In [ ]:
def construct_feature_matrix(img, coords, r=4):
    assert img.ndim == 3,\
        "Image dimension needs to be (C, Y, X)"
    
    n_nodes, n_chans = coords.shape[0], img.shape[0]
    ydim, xdim = img.shape[1:]
    features = np.zeros((n_nodes, n_chans))
    for i, (y, x) in enumerate(coords):
        
        indices = (slice(0, n_chans), 
                   slice(max(0, y-r), min(ydim-1, y+r)), 
                   slice(max(0, x-r), min(xdim-1, x+r)))
        features[i] = img[indices].sum((1, 2))

    return features
    

In [ ]:
features = [construct_feature_matrix(img, coords, r=8)
            for img, coords in zip(imgs, coords_list)]

In [ ]:
def disp_corr_features(features, labels=None, titles=None, ncols=4):
    n_slices = len(features)
    nrows = n_slices // ncols if n_slices % ncols == 0 else n_slices // ncols + 1

    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 3*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= n_slices:
                axes[r, c].axis('off')
                continue
            corr = np.corrcoef(features[idx].T)
            sns.heatmap(corr, mask=np.triu(corr),
                        xticklabels=labels, yticklabels=labels,
                        vmin=-0.3, vmax=0.3, square=True, 
                        ax=axes[r, c], cmap='coolwarm')
            
            if titles is not None:
                axes[r, c].set_title(titles[idx])
            idx += 1
    plt.tight_layout()
    plt.suptitle('Feature correlations (w/ Graph)', fontsize=30, y=1.02)
    plt.show()
    return None
            

In [ ]:
disp_corr_features(features, labels=cyif_chans, titles=filenames, ncols=6)

In [ ]:
# Save feature 
graph_path = '../data/cycif/graph/'
for f, feature in xzip(filenames, features):
    feature_df = pd.DataFrame(feature, columns=cyif_chans)
    feature_df.to_csv(os.path.join(graph_path, f.split('.')[0] + '.csv'), index=True)

#### H&E channels

### VGAE

2-layer dim-reduction, first trained on a single graph

In [ ]:
import torch
import torch.nn as nn

from tqdm import trange
from torch_geometric.nn import VGAE, GCNConv
from ml_collections import ConfigDict

In [ ]:
img_path = '../data/cycif/zonation_hires/'
graph_path = '../data/cycif/graph/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']

# Load images
imgs_rgb = [tifffile.imread(os.path.join(img_path, f))
            for f in filenames]

# Load graphs
Gs = [pickle.load(open(os.path.join(graph_path, f), 'rb'))
      for f in sorted(os.listdir(graph_path))
      if f[-3:] == 'pkl']

# Retrieve graph coords
coords_list = [get_coords_from_graph(G) for G in Gs]

In [ ]:
class GCNEncoder(nn.Module):
    def __init__(self, configs):
        super(GCNEncoder, self).__init__()
        self.layer1 = GCNConv(configs.c_in, configs.c_hidden)
        self.qz_mu = GCNConv(configs.c_hidden, configs.c_latent)
        self.qz_logstd = GCNConv(configs.c_hidden, configs.c_latent)

    def forward(self, x, edge_index):
        x = self.layer1(x, edge_index).relu()
        z_mu = self.qz_mu(x, edge_index)
        z_logstd = self.qz_logstd(x, edge_index)
        return z_mu, z_logstd
        

In [ ]:
def run_one_epoch():
    model.train()
    optimizer.zero_grad()
    z = model.encode(x, edge_index)
    reconst_loss = model.recon_loss(z, edge_index)
    kl_loss = model.kl_loss()
    loss = reconst_loss + (1 / n_nodes) * kl_loss
    loss.backward()
    optimizer.step()
    return float(loss), float(reconst_loss), float(kl_loss)


def train(n_epochs):
    losses = []
    nlls = []
    kls = []
    
    pbar = trange(n_epochs, desc='Training', leave=True)
    for epoch in pbar:
        loss, nll, kl = run_one_epoch()
        losses.append(loss)
        nlls.append(nll)
        kls.append(kl)

        pbar.set_postfix({'Training loss': '{:.3f}'.format(loss),
                          'NLL': '{:.3f}'.format(nll),
                          'KL': '{:.3f}'.format(kl)})
    pbar.close()
    return losses, nlls, kls


def test():
    model.eval()
    with torch.no_grad():
        z = model.encode(x, edge_index)
    return z.detach().cpu().numpy()

In [ ]:
# Model configs & initialization
model_configs = ConfigDict()
model_configs.c_in = 9
model_configs.c_hidden = 4
model_configs.c_latent = 1  # Learning heat directly?

model = VGAE(GCNEncoder(model_configs))
device = torch.device('cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
sample_features = pd.read_csv(os.path.join(graph_path, filenames[10].split('.')[0]+'.csv'), index_col=[0]).to_numpy()
sample_features = utils.znorm(sample_features)
sample_graph = Gs[10]
n_nodes = len(sample_graph)

x = torch.tensor(sample_features)
x = x.float()
edge_index = utils.nx_to_edge_index(sample_graph)

# model = model.to(device)
# x = x.float().to(device)
# edge_index = edge_index.to(device)

In [ ]:
n_epochs = 200
losses, nlls, kls = train(n_epochs)

In [ ]:
# cleanup for retrain
# del model, device, optimizer

In [ ]:
z = test()
plt.figure(figsize=(4, 2))
plt.hist(z, bins=30)
plt.show()

In [ ]:
cyif_chans = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68'
]

In [ ]:
img = imgs_rgb[10]
coords = coords_list[10]

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(21, 5))
ax1.imshow(img[1], cmap='magma', vmax=64)
ax1.axis('off')
ax1.set_title('GS (CV marker)')
ax2.imshow(img[2], cmap='magma', vmax=64)
ax2.axis('off')
ax2.set_title('CYP (PC marker)')
ax3.imshow(img[3], cmap='magma', vmax=64)
ax3.axis('off')
ax3.set_title('ASS1 (PP marker)')
qlow, qhigh = np.quantile(z, q=(0.1, 0.9))
im = ax4.scatter(coords[:, 1], coords[:, 0], s=1,
                 c=z, cmap='jet')
ax4.axis('off')
ax4.set_title('Learnt U')
plt.colorbar(im, ax=ax4)

plt.tight_layout()
plt.show()

In [ ]:
def disp_network_embedding(img, G, embedding, figsize=None, 
                           alpha=0.5, img_vmax=64, 
                           node_size=5, edge_width=1,
                           title=None):
    # coords = get_coords_from_graph(G)
    pos = {n: G.nodes[n]['pos'] for n in G.nodes}
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap='magma', alpha=alpha, vmax=img_vmax)
    # im = ax.scatter(coords[:, 1], coords[:, 0], s=node_size, c=embedding, cmap='jet')
    nx.draw_networkx_nodes(G, pos, node_color=embedding, node_size=node_size, cmap='jet', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=edge_width, ax=ax)
    ax.axis('off')    
    plt.colorbar(im, ax=ax, fraction=0.02, pad=1e-4)
    plt.tight_layout()
    plt.title(title, fontsize=20, y=0.95)
    plt.show()

In [ ]:
disp_network_embedding(img[1], sample_graph, embedding=z, alpha=0.25,
                       node_size=1, edge_width=0.2, figsize=(8, 8),
                       title='GS vs. Learnt U')

In [ ]:
disp_network_embedding(img[2], sample_graph, embedding=z, alpha=0.25,
                       node_size=1, edge_width=0.1, figsize=(8, 8),
                       title='CYP vs. Learnt U')
                       

In [ ]:
disp_network_embedding(img[3], sample_graph, embedding=z, alpha=0.25,
                       node_size=1, edge_width=0.1, figsize=(8, 8),
                       title='ASS vs. Learnt U')
                       

In [ ]:
disp_network_embedding(img[6], sample_graph, embedding=z, alpha=1,
                       node_size=1, edge_width=0.1, figsize=(8, 8),
                       title='CD45 vs. Learnt U')

VGAE: PCs as baseline

In [ ]:
# Baseline: PCA??
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pc = pca.fit_transform(sample_features)

disp_network_embedding(img[3], sample_graph, embedding=pc[:, 0], alpha=0.25,
                       node_size=1, edge_width=0.2, figsize=(8, 8),
                       title='ASS vs. PC2')
                       

In [ ]:
plt.figure(figsize=(3, 1.5))
plt.hist(z, bins=60, edgecolor='white')
plt.title('Learnt U')
plt.show()

In [ ]:
plt.figure(figsize=(3, 1.5))
plt.hist(pc[:, 0], bins=60, edgecolor='white')
plt.title('PC1')
plt.show()

In [ ]:
from scipy.stats import pearsonr
plt.scatter(z.squeeze(), pc[:, 0], s=1)
plt.xlabel('Learnt U')
plt.ylabel('PC1')
plt.title('R={}'.format(
    np.round(pearsonr(z.squeeze(), pc[:, 0])[0], 4)
))

---